In [ ]:
import torch
import torch.nn as nn

class BidirectionalLSTMModel(nn.Module):

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 512, num_layers: int = 2):

        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.output_dim = output_dim

        self.input_projection = nn.Linear(input_dim, hidden_dim)

        # Two-layer bidirectional LSTM network
        self.lstm = nn.LSTM(
            input_size=hidden_dim,        # Input to LSTM is from the projection layer
            hidden_size=hidden_dim,       # Size of the hidden state
            num_layers=num_layers,        # Number of stacked LSTMs
            batch_first=True,             # Input tensors are of shape (batch, seq_len, features)
            bidirectional=True            # Use a bidirectional LSTM
        )

        self.output_projection = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        projected_input = self.input_projection(x)

        lstm_output, _ = self.lstm(projected_input)
        
        final_output = lstm_output[:, -1, :]

        final_projection = self.output_projection(final_output)

        return final_projection

# --- Usage Example ---
l = 1
batch_size = 64
input_dim = 3
output_dim = 3
hidden_dim = 512

model = BidirectionalLSTMModel(
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_dim=hidden_dim
)

dummy_input = torch.randn(batch_size, l, input_dim)

output = model(dummy_input)

print(f"Model: \n{model}")
print(f"\nShape of input tensor: {dummy_input.shape}")
print(f"Shape of output tensor: {output.shape}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, train_path: str, label_path: str):

        with open(train_path,"r") as f:
            for line in f:
                train = eval(line)
        with open(label_path,"r") as f:
            for line in f:
                label = eval(line)
        
        self.samples = np.array(train, dtype=np.int32)
        self.labels = np.array(label, dtype=np.int32)

        if self.samples.shape[0] != self.labels.shape[0]:
            raise ValueError("The number of samples and labels must be equal.")
        if self.samples.shape[1] % 3 != 0:
            raise ValueError("The number of features must be a multiple of 3.")

        self.samples = torch.from_numpy(self.samples).to(torch.float32)
        self.labels = torch.from_numpy(self.labels).to(torch.float32)

        self.seq_len = self.samples.shape[1] // 3

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        label = self.labels[idx]

        sample = sample.view(self.seq_len, 3)

        return sample, label

if __name__ == "__main__":
    dataset = CustomDataset(train_path="train.data", label_path="label.data")

    data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

    input_dim = 3
    output_dim = 3
    hidden_dim = 512
    
    model = BidirectionalLSTMModel(input_dim=input_dim, output_dim=output_dim, hidden_dim=hidden_dim)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

    num_epochs = 30
    for epoch in range(num_epochs):
        model.train()
        for inputs, targets in data_loader:
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

    model.eval()

    random_indices = np.random.choice(len(dataset), 50, replace=False)

    for i, idx in enumerate(random_indices):
        sample_tensor, true_label = dataset[idx]

        with torch.no_grad():
            predicted_output = model(sample_tensor.unsqueeze(0))



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

class BidirectionalLSTMModel(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 512, num_layers: int = 2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.output_dim = output_dim
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.output_projection = nn.Linear(hidden_dim * 2, output_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        projected_input = self.input_projection(x)
        lstm_output, _ = self.lstm(projected_input)
        final_output = lstm_output[:, -1, :]
        final_projection = self.output_projection(final_output)
        return final_projection

class CustomDataset(Dataset):
    def __init__(self, train_path: str, label_path: str):
        with open(train_path, "r") as f:
            for line in f:
                train_lines = eval(line)
        with open(label_path, "r") as f:
            for line in f:
                label_lines = eval(line)
                
        self.samples = np.array(train_lines, dtype=np.int32)
        self.labels = np.array(label_lines, dtype=np.int32)
        
        if self.samples.shape[0] != self.labels.shape[0]:
            raise ValueError("The number of samples and labels must be equal.")
        if self.samples.ndim != 2 or self.labels.ndim != 2:
            raise ValueError("The data should be a 2D array.")
        if self.samples.shape[1] % 3 != 0:
            raise ValueError("The number of features must be a multiple of 3.")
            
        self.samples = torch.from_numpy(self.samples).to(torch.float32)
        self.labels = torch.from_numpy(self.labels).to(torch.float32)
        
        self.seq_len = self.samples.shape[1] // 3
        
    def __len__(self):
        return self.samples.shape[0]
    s
    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        label = self.labels[idx]
        sample = sample.view(self.seq_len, 3)
        return sample, label

if __name__ == "__main__":

    dataset = CustomDataset(train_path="train.data", label_path="label.data")
    data_loader = DataLoader(dataset, batch_size=64, shuffle=True)
    input_dim = 3
    output_dim = 3
    hidden_dim = 512
    
    model = BidirectionalLSTMModel(input_dim=input_dim, output_dim=output_dim, hidden_dim=hidden_dim)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        for inputs, targets in data_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

    model.eval()
    random_indices = np.random.choice(len(dataset), 50, replace=False)

    for i, idx in enumerate(random_indices):
        sample_tensor, true_label = dataset[idx]
        
        with torch.no_grad():
            predicted_output = model(sample_tensor.unsqueeze(0))
            
            rounded_output = torch.round(predicted_output)
